# Frozen external dehazing baseline comparison (Kaggle, inference only)

Compare the three fixed ECA-LDNet Stage 4 checkpoints and their fixed 1/3 output average against **ED-Former trained on RESIDE-OTS**, using identical paired images and 512×512 RGB inputs. O-HAZE all 45 pairs is primary. Optional Dense-Haze uses pooled 55-pair release as secondary context, not as an official challenge-only score.

ED-Former official repository releases the RESIDE-OTS checkpoint and gives its SHA-256. Its README reports a separate zero-shot O-HAZE result; this notebook does **not** reproduce that protocol. We use this project's radiometric loader, fixed-square resize, clipping, and common metric code so paired outputs share one protocol. ECA-LDNet uses RESIDE/ITS training data while ED-Former checkpoint uses RESIDE-OTS, so this is a fixed-checkpoint comparison, not a controlled architecture-only test. Do not infer component effects or state-of-the-art status.

No training, fine-tuning, image-specific selection, or parameter tuning. The notebook records the external code commit and checkpoint hash. Internet must be enabled in Kaggle to fetch the official implementation and checkpoint. The ED-Former repository says evaluation and citation are permitted but provides no general software license; use it for evaluation only and do not redistribute its code/checkpoint.

Sources: [ED-Former official repository](https://github.com/2697166190a-beep/ED-Former), [paper DOI](https://doi.org/10.1016/j.image.2026.117634).

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import platform
import subprocess
import sys
import time
import urllib.request
import zipfile

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import torch

# Pin model-code dependencies without replacing Kaggle's CUDA-enabled Torch/TorchVision.
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "timm==0.9.16", "pytorch_wavelets==1.3.0"
], check=True)

REPO_URL = "https://github.com/2697166190a-beep/ED-Former.git"
REPO_DIR = Path("/kaggle/working/ED-Former")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
sys.path.insert(0, str(REPO_DIR))
from ED_Former import ED_Former

PACKAGE = Path("/kaggle/input/datasets/randomguy28/reviewer-test-packge/reviewer_test_package")
OHAZE_ROOT = Path("/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY")
DENSE_ROOT = Path("/kaggle/input/datasets/lucky02655/den-haze-test")
IMAGE_SIZE = 512
SEEDS = (17, 43, 73)
ENSEMBLE_WEIGHTS = {seed: 1.0 / len(SEEDS) for seed in SEEDS}
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_SEED = 20261014
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
METRICS = ("psnr", "ssim", "ms_ssim", "mae", "deltaE2000")
BASELINE_LABEL = "ED-Former (RESIDE-OTS)"
EXPECTED_EDFORMER_SHA256 = "ea7d7e3988ec98daea7b23f216bca7597fa987675c4ac7df5c001116459aef3a"
EDFORMER_URL = "https://raw.githubusercontent.com/2697166190a-beep/ED-Former/main/saved_models/outdoor/ed-former.pth"
EDFORMER_CHECKPOINT = REPO_DIR / "saved_models/outdoor/ed-former.pth"
if not EDFORMER_CHECKPOINT.is_file():
    urllib.request.urlretrieve(EDFORMER_URL, EDFORMER_CHECKPOINT)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

EDFORMER_SHA256 = sha256_file(EDFORMER_CHECKPOINT)
assert EDFORMER_SHA256 == EXPECTED_EDFORMER_SHA256, (
    f"ED-Former checkpoint hash mismatch: {EDFORMER_SHA256}"
)

assert PACKAGE.is_dir(), f"Missing ECA-LDNet reviewer package: {PACKAGE}"
assert (OHAZE_ROOT / "hazy").is_dir(), f"Missing O-HAZE hazy folder: {OHAZE_ROOT / 'hazy'}"
assert (OHAZE_ROOT / "GT").is_dir(), f"Missing O-HAZE GT folder: {OHAZE_ROOT / 'GT'}"
SCRIPT_ROOT = PACKAGE / "code"
sys.path.insert(0, str(SCRIPT_ROOT))
from eca_ldnet.data import get_gt_path, load_image_pair
from eca_ldnet.evaluate import predict_full
from eca_ldnet.model import load_pretrained

ECA_CHECKPOINTS = {
    seed: PACKAGE / f"models/full/seed{seed}/stage4_best.keras"
    for seed in SEEDS
}
missing = [str(path) for path in ECA_CHECKPOINTS.values() if not path.is_file()]
assert not missing, f"Missing fixed ECA-LDNet checkpoints: {missing}"

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

RUN_ID = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns() % 1_000_000_000:09d}"
OUTPUT_ROOT = Path("/kaggle/working/frozen_edformer_comparison") / f"run_{RUN_ID}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("ED-Former repository commit:", REPO_COMMIT)
print("ED-Former checkpoint SHA-256:", EDFORMER_SHA256)
print("PyTorch device:", DEVICE)
print("TensorFlow GPUs:", [device.name for device in tf.config.list_physical_devices("GPU")])
print("Output directory:", OUTPUT_ROOT)

In [ ]:
def collect_pairs(root, expected_count=None):
    hazy_dir, gt_dir = root / "hazy", root / "GT"
    if not hazy_dir.is_dir() or not gt_dir.is_dir():
        return [], {"status": "skipped", "reason": "hazy or GT directory missing", "root": str(root)}
    files = sorted(path for path in hazy_dir.iterdir() if path.suffix.lower() in IMAGE_EXTS)
    pairs, unmatched = [], []
    for hazy_path in files:
        gt_path = get_gt_path(hazy_path.name, str(gt_dir))
        if gt_path is None:
            unmatched.append(hazy_path.name)
        else:
            pairs.append({"image": hazy_path.name, "hazy_path": str(hazy_path), "gt_path": str(gt_path)})
    info = {"status": "available", "root": str(root), "hazy_files": len(files),
            "matched_pairs": len(pairs), "unmatched_hazy": unmatched}
    if expected_count is not None and len(pairs) != expected_count:
        info.update({"status": "skipped", "reason": f"expected {expected_count} pairs, found {len(pairs)}"})
        return [], info
    if unmatched:
        info.update({"status": "skipped", "reason": "one or more hazy images lack matched ground truth"})
        return [], info
    return pairs, info


def load_pairs(pairs, dataset_name, required=False):
    loaded, unreadable = [], []
    for pair in pairs:
        hazy, clean = load_image_pair(
            pair["hazy_path"], pair["gt_path"], size=IMAGE_SIZE, preprocessing="radiometric"
        )
        if hazy is None or clean is None or hazy.shape != clean.shape:
            unreadable.append(pair["image"])
        else:
            loaded.append({"image": pair["image"], "hazy": hazy, "clean": clean,
                           "hazy_path": pair["hazy_path"], "gt_path": pair["gt_path"]})
    if unreadable and required:
        raise RuntimeError(f"Unreadable/mismatched {dataset_name} pairs: {unreadable}")
    return loaded, unreadable

ohaze_pairs, ohaze_info = collect_pairs(OHAZE_ROOT, expected_count=45)
assert ohaze_info["status"] == "available", f"O-HAZE must have all 45 pairs: {ohaze_info}"
dense_pairs, dense_info = collect_pairs(DENSE_ROOT, expected_count=55)
if dense_info["status"] == "skipped":
    print("Dense-Haze skipped:", dense_info["reason"])
PAIR_SETS = {"O-HAZE": ohaze_pairs}
if dense_info["status"] == "available":
    PAIR_SETS["Dense-Haze pooled release (55)"] = dense_pairs
DATASETS, UNREADABLE = {}, {}
for dataset_name, pairs in PAIR_SETS.items():
    loaded, unreadable = load_pairs(pairs, dataset_name, required=(dataset_name == "O-HAZE"))
    if unreadable and dataset_name != "O-HAZE":
        dense_info.update({"status": "skipped", "reason": f"unreadable/mismatched pairs: {unreadable}"})
        print("Dense-Haze skipped:", dense_info["reason"])
        continue
    if dataset_name == "O-HAZE":
        assert len(loaded) == 45, f"Expected 45 readable O-HAZE pairs, found {len(loaded)}"
    DATASETS[dataset_name] = loaded
    UNREADABLE[dataset_name] = unreadable
    print(f"{dataset_name}: {len(loaded)} readable pairs")

In [ ]:
try:
    from skimage.color import deltaE_ciede2000, rgb2lab
    HAS_DELTA_E = True
except ImportError:
    HAS_DELTA_E = False
    print("scikit-image missing: CIEDE2000 will be NaN; other metrics remain available.")


def score_prediction(prediction, clean):
    pred = np.clip(np.asarray(prediction, dtype=np.float32), 0.0, 1.0)
    gt = np.clip(np.asarray(clean, dtype=np.float32), 0.0, 1.0)
    diff = pred.astype(np.float64) - gt.astype(np.float64)
    result = {
        "psnr": float(tf.image.psnr(pred[None], gt[None], 1.0).numpy().reshape(-1)[0]),
        "ssim": float(tf.image.ssim(pred[None], gt[None], 1.0).numpy().reshape(-1)[0]),
        "mae": float(np.mean(np.abs(diff))),
    }
    try:
        result["ms_ssim"] = float(tf.image.ssim_multiscale(pred[None], gt[None], 1.0).numpy().reshape(-1)[0])
    except (tf.errors.InvalidArgumentError, ValueError):
        result["ms_ssim"] = float("nan")
    result["deltaE2000"] = float(np.mean(deltaE_ciede2000(rgb2lab(pred), rgb2lab(gt)))) if HAS_DELTA_E else float("nan")
    return result


def load_edformer(checkpoint_path):
    # Official test.py loads a plain state dict and strips optional DataParallel prefixes.
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    state = checkpoint.get("state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    state = {key.removeprefix("module."): value for key, value in state.items()}
    model = ED_Former()
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE).eval()

print("CIEDE2000 available:", HAS_DELTA_E)

## Run fixed-checkpoint inference

ED-Former expects RGB tensors in `[-1,1]`; its official test code clips outputs to `[-1,1]` and maps back to `[0,1]`. The shared loader square-resizes hazy and ground-truth images to 512×512. We run ED-Former first, release its GPU memory, then score the fixed ECA-LDNet checkpoints and equal-weight ensemble. No weights are tuned on these images.

In [ ]:
inference_started = time.perf_counter()
external_predictions = {}

edformer = load_edformer(EDFORMER_CHECKPOINT)
for dataset_name, records in DATASETS.items():
    predictions = []
    print(f"ED-Former inference: {dataset_name}, n={len(records)}")
    with torch.inference_mode():
        for index, record in enumerate(records, start=1):
            rgb = np.asarray(record["hazy"], dtype=np.float32)
            x = torch.from_numpy(np.transpose(rgb * 2.0 - 1.0, (2, 0, 1))[None]).to(DEVICE)
            output = edformer(x).clamp(-1.0, 1.0)
            prediction = (output * 0.5 + 0.5)[0].permute(1, 2, 0).cpu().numpy().astype(np.float32)
            predictions.append(prediction)
            if index % 10 == 0 or index == len(records):
                print(f"  {index}/{len(records)}")
    external_predictions[dataset_name] = predictions

del edformer
torch.cuda.empty_cache()

score_rows = []
eca_checkpoint_hashes = {}
for dataset_name, records in DATASETS.items():
    print(f"ECA-LDNet inference: {dataset_name}, n={len(records)}")
    seed_predictions = {seed: [] for seed in SEEDS}
    for seed, checkpoint in ECA_CHECKPOINTS.items():
        eca_checkpoint_hashes[seed] = {"path": str(checkpoint), "sha256": sha256_file(checkpoint)}
        model = load_pretrained(str(checkpoint), compile=False)
        shape = model.input_shape
        if shape[1] is not None and shape[1] != IMAGE_SIZE:
            raise ValueError(f"ECA-LDNet seed {seed} expects height {shape[1]}, not {IMAGE_SIZE}")
        if shape[2] is not None and shape[2] != IMAGE_SIZE:
            raise ValueError(f"ECA-LDNet seed {seed} expects width {shape[2]}, not {IMAGE_SIZE}")
        for index, record in enumerate(records):
            prediction = predict_full(model, record["hazy"])
            seed_predictions[seed].append(prediction)
            score_rows.append({"dataset": dataset_name, "image": record["image"],
                               "method": f"ECA-LDNet seed {seed}", "seed": seed,
                               **score_prediction(prediction, record["clean"])})
        del model
        tf.keras.backend.clear_session()

    for index, record in enumerate(records):
        predictions = np.stack([seed_predictions[seed][index] for seed in SEEDS], axis=0)
        ensemble = np.tensordot(
            np.asarray([1.0 / len(SEEDS)] * len(SEEDS), dtype=np.float32), predictions, axes=(0, 0)
        )
        ensemble = np.clip(ensemble, 0.0, 1.0).astype(np.float32)
        score_rows.append({"dataset": dataset_name, "image": record["image"],
                           "method": "ECA-LDNet uniform ensemble", "seed": "all",
                           **score_prediction(ensemble, record["clean"])})
        score_rows.append({"dataset": dataset_name, "image": record["image"],
                           "method": BASELINE_LABEL, "seed": "fixed RESIDE-OTS",
                           **score_prediction(external_predictions[dataset_name][index], record["clean"])})
    del seed_predictions

scores = pd.DataFrame(score_rows)
expected_rows = sum(len(records) for records in DATASETS.values()) * (len(SEEDS) + 2)
assert len(scores) == expected_rows, f"Expected {expected_rows} rows; found {len(scores)}"
scores.to_csv(OUTPUT_ROOT / "per_image_scores.csv", index=False)
print(f"Saved {len(scores):,} rows to per_image_scores.csv")
print(f"Inference and scoring elapsed: {(time.perf_counter() - inference_started) / 60:.1f} min")

In [ ]:
# Descriptive summaries and image-paired ECA-LDNet minus ED-Former contrasts.
summary_rows = []
for (dataset, method), group in scores.groupby(["dataset", "method"], sort=False):
    for metric in METRICS:
        values = group[metric].to_numpy(dtype=float)
        finite = values[np.isfinite(values)]
        summary_rows.append({"dataset": dataset, "method": method, "metric": metric,
                             "n_images": int(len(finite)),
                             "mean": float(np.mean(finite)) if len(finite) else np.nan,
                             "std": float(np.std(finite, ddof=1)) if len(finite) > 1 else np.nan})
method_summary = pd.DataFrame(summary_rows)
method_summary.to_csv(OUTPUT_ROOT / "dataset_method_summary.csv", index=False)

paired_rows = []
ECA_METHODS = [f"ECA-LDNet seed {seed}" for seed in SEEDS] + ["ECA-LDNet uniform ensemble"]
for dataset in DATASETS:
    external = scores[(scores["dataset"] == dataset) & (scores["method"] == BASELINE_LABEL)].set_index("image")
    for eca_method in ECA_METHODS:
        eca = scores[(scores["dataset"] == dataset) & (scores["method"] == eca_method)].set_index("image")
        if set(external.index) != set(eca.index):
            raise RuntimeError(f"Image pairing mismatch for {dataset} / {eca_method}")
        for image in external.index:
            row = {"dataset": dataset, "image": image, "ECA_method": eca_method,
                   "external_baseline": BASELINE_LABEL}
            for metric in METRICS:
                row[f"ECA_{metric}"] = eca.loc[image, metric]
                row[f"EDFormer_{metric}"] = external.loc[image, metric]
                row[f"delta_ECA_minus_EDFormer_{metric}"] = eca.loc[image, metric] - external.loc[image, metric]
            paired_rows.append(row)
paired = pd.DataFrame(paired_rows)
paired.to_csv(OUTPUT_ROOT / "paired_eca_vs_edformer_per_image.csv", index=False)

DIRECTION = {"psnr": "higher", "ssim": "higher", "ms_ssim": "higher",
             "mae": "lower", "deltaE2000": "lower"}
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows = []
for (dataset, eca_method), group in paired.groupby(["dataset", "ECA_method"], sort=False):
    for metric in METRICS:
        values = group[f"delta_ECA_minus_EDFormer_{metric}"].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if not len(values):
            continue
        draws = rng.integers(0, len(values), size=(BOOTSTRAP_SAMPLES, len(values)))
        boot_means = values[draws].mean(axis=1)
        better = values > 0 if DIRECTION[metric] == "higher" else values < 0
        bootstrap_rows.append({
            "dataset": dataset, "ECA_method": eca_method, "metric": metric,
            "n_paired_images": int(len(values)), "mean_delta_ECA_minus_EDFormer": float(np.mean(values)),
            "bootstrap_ci95_low": float(np.quantile(boot_means, 0.025)),
            "bootstrap_ci95_high": float(np.quantile(boot_means, 0.975)),
            "ECA_win_fraction_images": float(np.mean(better)),
            "direction_for_win": DIRECTION[metric],
            "bootstrap_unit": "paired image; conditional on fixed checkpoints",
        })
bootstrap_summary = pd.DataFrame(bootstrap_rows)
bootstrap_summary.to_csv(OUTPUT_ROOT / "paired_bootstrap_summary.csv", index=False)
print("Mean PSNR by method:")
print(method_summary[method_summary["metric"] == "psnr"].to_string(index=False))
print("Ensemble versus ED-Former PSNR:")
print(bootstrap_summary[(bootstrap_summary["metric"] == "psnr") & (bootstrap_summary["ECA_method"] == "ECA-LDNet uniform ensemble")].to_string(index=False))

In [ ]:
# Plots: method means and paired ensemble-minus-ED-Former PSNR differences.
PLOT_METHODS = [f"ECA-LDNet seed {seed}" for seed in SEEDS] + ["ECA-LDNet uniform ensemble", BASELINE_LABEL]
COLORS = ["#4c78a8", "#72a0c1", "#9dc2d9", "#d17a22", "#777777"]
datasets_present = list(DATASETS)
fig, axes = plt.subplots(1, len(datasets_present), figsize=(7 * len(datasets_present), 4), squeeze=False)
for ax, dataset in zip(axes[0], datasets_present):
    part = method_summary[(method_summary["dataset"] == dataset) & (method_summary["metric"] == "psnr")]
    part = part.set_index("method").reindex(PLOT_METHODS).dropna(subset=["mean"])
    ax.bar(part.index, part["mean"], color=COLORS[:len(part)])
    ax.set_title(dataset)
    ax.set_ylabel("Mean PSNR (dB)")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "mean_psnr_by_method.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

ensemble_rows = bootstrap_summary[(bootstrap_summary["ECA_method"] == "ECA-LDNet uniform ensemble") & (bootstrap_summary["metric"] == "psnr")]
fig, axes = plt.subplots(1, len(datasets_present), figsize=(6 * len(datasets_present), 4), squeeze=False)
for ax, dataset in zip(axes[0], datasets_present):
    part = ensemble_rows[ensemble_rows["dataset"] == dataset]
    mean = part["mean_delta_ECA_minus_EDFormer"].to_numpy()
    low = part["bootstrap_ci95_low"].to_numpy()
    high = part["bootstrap_ci95_high"].to_numpy()
    ax.errorbar([0], mean, yerr=np.vstack([mean - low, high - mean]), fmt="o", capsize=5, color="#355c7d")
    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.set_xticks([0], ["ECA ensemble − ED-Former"])
    ax.set_ylabel("Paired PSNR difference (dB)")
    ax.set_title(dataset)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "paired_ensemble_vs_edformer_psnr.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
dataset_manifest = {
    "O-HAZE": {"root": str(OHAZE_ROOT), "status": ohaze_info["status"],
               "readable_pairs": len(DATASETS["O-HAZE"]),
               "images": [row["image"] for row in DATASETS["O-HAZE"]]},
    "Dense-Haze pooled release": dense_info,
}
if "Dense-Haze pooled release (55)" in DATASETS:
    dataset_manifest["Dense-Haze pooled release"].update({
        "readable_pairs": len(DATASETS["Dense-Haze pooled release (55)" ]),
        "images": [row["image"] for row in DATASETS["Dense-Haze pooled release (55)" ]],
        "interpretation": "pooled 55-pair release, not official challenge-only evaluation",
    })
manifest = {
    "experiment": "fixed-checkpoint ECA-LDNet versus ED-Former reference",
    "scope": "inference-only paired comparison; no training, fine-tuning, or evaluation-set model selection",
    "resolution": [IMAGE_SIZE, IMAGE_SIZE],
    "color_order": "RGB",
    "input_preprocessing": "radiometric RGB [0,1], fixed-square evaluator loader",
    "output_clipping": {"ECA-LDNet": "[0,1]", "ED-Former": "clamp [-1,1], then map to [0,1] per official test code"},
    "ECA_LDNet_checkpoints": eca_checkpoint_hashes,
    "ED_Former": {
        "repository": REPO_URL, "commit": REPO_COMMIT,
        "checkpoint": "saved_models/outdoor/ed-former.pth", "training_release": "RESIDE-OTS",
        "sha256": EDFORMER_SHA256, "expected_sha256": EXPECTED_EDFORMER_SHA256,
    },
    "ECA_ensemble": {"rule": "arithmetic mean of three clipped RGB outputs",
                      "weights_by_seed": ENSEMBLE_WEIGHTS},
    "metrics": list(METRICS),
    "metric_conventions": {
        "psnr_ssim_ms_ssim": "TensorFlow tf.image, max_val=1.0",
        "mae": "mean absolute error over RGB values",
        "deltaE2000": "mean CIEDE2000 after skimage rgb2lab on clipped RGB [0,1]",
    },
    "datasets": dataset_manifest,
    "bootstrap": {"samples": BOOTSTRAP_SAMPLES, "seed": BOOTSTRAP_SEED,
                   "unit": "paired image", "interpretation": "conditional on fixed checkpoints and datasets"},
    "limitations": [
        "external comparison is descriptive and not architecture-only because training data differ",
        "does not establish state-of-the-art performance or component effects",
        "ED-Former official O-HAZE score uses a different evaluation pipeline and is not directly compared here",
        "Dense-Haze is pooled-release context and remains secondary",
        "no latency claim beyond this Kaggle run; TensorFlow and PyTorch share one visible accelerator sequentially",
    ],
    "python": platform.python_version(), "tensorflow": tf.__version__,
    "torch": torch.__version__, "opencv": cv2.__version__,
    "gpu_devices": [device.name for device in tf.config.list_physical_devices("GPU")],
    "torch_device": str(DEVICE), "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with (OUTPUT_ROOT / "run_manifest.json").open("w", encoding="utf-8") as stream:
    json.dump(manifest, stream, indent=2)
print("Run outputs:", sorted(path.name for path in OUTPUT_ROOT.iterdir()))

## Package this run's outputs

Run this final cell after inference, analysis, plots, and manifest generation. It recursively archives every output file created in this run and verifies ZIP integrity.

In [ ]:
ZIP_PATH = Path("/kaggle/working/frozen_edformer_comparison.zip")
output_files = sorted(path for path in OUTPUT_ROOT.rglob("*") if path.is_file())
if not output_files:
    raise RuntimeError(f"No output files found under {OUTPUT_ROOT}; run analysis cells first.")
expected_members = {path.relative_to(OUTPUT_ROOT).as_posix() for path in output_files}
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_files:
        archive.write(path, arcname=path.relative_to(OUTPUT_ROOT).as_posix())
with zipfile.ZipFile(ZIP_PATH, mode="r") as archive:
    archived_members = set(archive.namelist())
    damaged_member = archive.testzip()
if archived_members != expected_members:
    raise RuntimeError(f"ZIP member mismatch; missing={sorted(expected_members - archived_members)}, unexpected={sorted(archived_members - expected_members)}")
if damaged_member is not None:
    raise RuntimeError(f"ZIP integrity check failed at {damaged_member}")
print(f"Verified {len(expected_members)} output files in {ZIP_PATH}")
print(f"Archive size: {ZIP_PATH.stat().st_size / (1024 * 1024):.2f} MiB")
for member in sorted(archived_members):
    print(" -", member)